# Signal — Fingerprint the Exact-0.01 BTC Order

Monitors the TrueMarkets DEPTH feed for the **0.01 BTC** order that `peak_clean` /
`integrated_clean` snipe — the biggest player in the book — and cross-references every
event against **Binance and Coinbase** mids to work out what its strategy actually is.

**Working model** (the claims this notebook confirms or falsifies):
1. it is always posted at its **own price level**, the **only order** there — so a level
   showing exactly 0.01 IS the order;
2. it **pops in and out** — placed, then cancelled, over and over;
3. when exposed, **the market moves against it** (the sniper's entry thesis);
4. **[the CB-peg hypothesis]** it posts **at the Coinbase price**, on the **ask side when
   Coinbase is above the TM mid** and the **bid side when Coinbase is below** — a
   one-sided cross-venue fair-value quoter.

**Events** (all logged to a timestamped CSV in `logs/`):

| Event | Meaning |
|---|---|
| `APPEAR` | a level became exactly 0.01 — `own level` if the level was empty before (claim 1 holds), `existing level!` if it was a residual (claim 1 violated) |
| `CANCEL?` | tracked level wiped without the book crossing it — looks like a pull (claim 2) |
| `FILL?` | tracked level wiped while the book crossed it — someone (us?) took it |
| `JOINED` / `PARTIAL` | level grew past / shrank below 0.01 — claim 1 violated |
| `THESIS` | TM-mid and external-fair drift 1s/5s/15s after each APPEAR, signed so **positive = market moved against the order** (claim 3) |
| `STALE` | ⚠ abandoned-order alarm: a tracked 0.01 that is ≥ `STALE_AFTER_SECS` old (they normally cancel in ~5s) AND mispriced vs CB by ≥ `STALE_EDGE_USD` — a broken-cancel leftover, directly exploitable by `main.ipynb`'s stale snipe |

**Strategy fingerprint** — each metric discriminates between hypotheses:

- **CB peg** (`peg_cb` = posting price − Coinbase mid, raw). If claim 4 holds, this
  histogram collapses onto 0. The header shows the median |peg| and the fraction within
  2 ticks.
- **Side rule** (`side_rule_ok` = did it post an ask exactly when CB mid > TM mid, a bid
  when below?). A hit rate near 100% confirms the one-sided fair-value-quoter read.
- **Posting edge vs external fair** (`edge_cb`/`edge_bn`, signed: + = behind fair,
  − = through it). A CB-pegged quoter shows edge ≈ 0 by construction.
- **Pre-post fair drift** (`pre_toward_ext`: external drift in the 1s *before* an APPEAR,
  signed toward the order's price). Consistently negative → it re-posts right after fair
  moves away from its side.
- **Lifetime fair drift at pull** (`life_toward_ext`, recorded at CANCEL?). Consistently
  positive → it pulls *because* fair is approaching it — its cancels are driven by the
  external feed, and sniping it is a race against that feed's latency.
- **THESIS drift in external terms** — if TM mid moves "against" the order but the external
  fair doesn't, the drift is just TM converging to the outside market (basis convergence),
  not fresh alpha.

All external-drift metrics use a **reference feed: Binance when it's alive, otherwise
Coinbase** (`ext_venue` in the CSV says which was used per event). `binance.com`
websockets are geo-blocked from US IPs — with the default URL the reference will
usually be Coinbase; a `binance.us` variant is provided in the parameters cell.
BTC-PYUSD vs USD/USDT introduces a small stable basis; the basis panel shows it directly.

In [ ]:
import asyncio
import csv
import json
import math
import statistics
import time
from collections import deque
from datetime import datetime
from pathlib import Path

import websockets
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display, clear_output


In [ ]:
TARGET_SIZE_STR = '0.01'    # exact string to match from WebSocket
TARGET_SIZE     = 0.01      # float version
SIZE_TOL        = 1e-9      # tolerance for float comparisons
TICK_SIZE       = 0.1
DRIFT_HORIZONS  = (1.0, 5.0, 15.0)   # secs after APPEAR to measure mid drift (thesis check)
PRE_WINDOW_SECS = 1.0       # external-fair lookback before an event ("did fair just move?")
EXT_STALE_SECS  = 5.0       # external feed older than this is ignored, never fabricated
PEG_CLOSE_TICKS = 2         # "posted at CB" = within this many ticks of Coinbase mid
STALE_AFTER_SECS = 30.0     # tracked 0.01 older than this (they cancel in ~5s) ...
STALE_EDGE_USD   = 25.0     # ...AND mispriced vs CB by this much -> abandoned-order ALERT

WS_URL          = 'wss://api.truex.co/api/v1'
SYMBOL          = 'BTC-PYUSD'
BINANCE_WS_URL  = 'wss://stream.binance.com:9443/ws/btcusdt@bookTicker'
# US-hosted? binance.com blocks US IPs — use:
# BINANCE_WS_URL = 'wss://stream.binance.us:9443/ws/btcusdt@bookTicker'
COINBASE_WS_URL = 'wss://ws-feed.exchange.coinbase.com'
CB_PRODUCT      = 'BTC-USD'

HISTORY_LEN     = 600
EVENT_DISPLAY   = 40
LEVELS_DISPLAY  = 8           # levels to show in per-level breakdown

LOG_DIR = Path('..') / 'logs'   # relative to notebooks/
LOG_DIR.mkdir(exist_ok=True)

### External reference feeds (Binance + Coinbase best bid/ask)

In [ ]:
class ExtFeed:
    """Best bid/ask from an external venue, with a short mid history so we can
    ask "where was fair X seconds ago?" around each 0.01 event."""

    def __init__(self, name):
        self.name     = name
        self.bid      = None
        self.ask      = None
        self.last_msg = None
        self.hist     = deque(maxlen=4000)   # (wall_time, mid)

    @property
    def mid(self):
        return (self.bid + self.ask) / 2 if (self.bid and self.ask) else None

    @property
    def fresh(self):
        return self.last_msg is not None and time.time() - self.last_msg <= EXT_STALE_SECS

    def _note(self, bid, ask):
        self.bid, self.ask = bid, ask
        self.last_msg = time.time()
        m = self.mid
        if m:
            self.hist.append((self.last_msg, m))

    def mid_at(self, t):
        """Most recent mid at or before wall time t (None if no data that old)."""
        out = None
        for ht, hm in self.hist:
            if ht <= t:
                out = hm
            else:
                break
        return out

    def drift_since(self, t0):
        m_now, m_then = self.mid, self.mid_at(t0)
        return (m_now - m_then) if (m_now is not None and m_then is not None) else None


class BinanceFeed(ExtFeed):
    """bookTicker stream: every best bid/ask change, no subscription message needed."""

    def __init__(self, url=BINANCE_WS_URL):
        super().__init__('binance')
        self.url = url

    async def run(self):
        backoff = 1
        while True:
            try:
                async with websockets.connect(self.url) as ws:
                    backoff = 1
                    async for raw in ws:
                        try: m = json.loads(raw)
                        except Exception: continue
                        b, a = m.get('b'), m.get('a')
                        if b and a:
                            self._note(float(b), float(a))
            except asyncio.CancelledError:
                raise
            except Exception as e:
                print(f'Binance feed error: {e}')
            await asyncio.sleep(backoff)
            backoff = min(backoff * 2, 30)


class CoinbaseFeed(ExtFeed):
    def __init__(self, url=COINBASE_WS_URL, product=CB_PRODUCT):
        super().__init__('coinbase')
        self.url     = url
        self.product = product

    async def run(self):
        backoff = 1
        while True:
            try:
                async with websockets.connect(self.url) as ws:
                    backoff = 1
                    await ws.send(json.dumps({
                        'type': 'subscribe',
                        'product_ids': [self.product],
                        'channels': ['ticker'],
                    }))
                    async for raw in ws:
                        try: m = json.loads(raw)
                        except Exception: continue
                        if m.get('type') == 'ticker':
                            b, a = m.get('best_bid'), m.get('best_ask')
                            if b and a:
                                self._note(float(b), float(a))
            except asyncio.CancelledError:
                raise
            except Exception as e:
                print(f'Coinbase feed error: {e}')
            await asyncio.sleep(backoff)
            backoff = min(backoff * 2, 30)

In [ ]:
class SignalTracker:
    """
    Tracks the exact-0.01 BTC order in the TM book and fingerprints it against
    external fair prices.

    Model: the order is always posted at its OWN price level and is the only
    order there, so detection is simply "a level whose displayed qty is exactly
    0.01". The JOINED / PARTIAL / `existing level!` annotations exist to
    FALSIFY that model — if they stay at zero, the model holds.

    Sign conventions (all "toward/against the order", so positive supports the
    corresponding hypothesis regardless of side):
      edge_*          + = posted BEHIND external fair (has edge), − = through it
      peg_cb          raw price − Coinbase mid (0 = posted AT Coinbase)
      side_rule_ok    True = ask posted with CB above TM mid, or bid with CB below
      pre_toward_ext  + = fair moved TOWARD the order's price just before it posted
      life_toward_ext + = fair moved TOWARD it during its life (recorded at vanish)
      THESIS drifts   + = market moved AGAINST the order after it appeared
    """

    _CSV_COLS = [
        'timestamp', 'elapsed_s', 'side', 'event',
        'price', 'total_qty', 'fresh', 'at_bbo', 'dist_bbo_ticks',
        'duration_s', 'gap_s',
        'mid', 'dist_mid_abs', 'dist_mid_pct',
        'bn_mid', 'cb_mid', 'ext_venue', 'tm_ref_basis',
        'peg_cb', 'side_rule_ok',
        'edge_bn', 'edge_cb', 'pre_drift_ext', 'pre_toward_ext',
        'life_drift_ext', 'life_toward_ext',
        'drift_1s', 'drift_5s', 'drift_15s',
        'ext_drift_1s', 'ext_drift_5s', 'ext_drift_15s',
    ]

    def __init__(self, log_path, ext=None):
        self._bids   = {}    # price -> float qty (aggregate)
        self._asks   = {}
        self.ext     = ext or {}          # {'binance': ExtFeed, 'coinbase': ExtFeed}
        self.tracked = {'bid': {}, 'ask': {}}   # price -> {t0, at_bbo, fresh, mid0, ref0, ref_name}
        self.last_vanish    = {'bid': None, 'ask': None}   # for re-post cadence
        self.pending_thesis = []   # APPEARs whose drift horizons haven't matured
        self.thesis_done    = []
        self.events   = deque(maxlen=2000)
        self.ts       = deque(maxlen=HISTORY_LEN)
        self.mids     = deque(maxlen=HISTORY_LEN)
        self.bn_mids  = deque(maxlen=HISTORY_LEN)
        self.cb_mids  = deque(maxlen=HISTORY_LEN)
        self.basis    = deque(maxlen=HISTORY_LEN)   # TM mid - reference fair
        self.t_bid_px = deque(maxlen=HISTORY_LEN)
        self.t_ask_px = deque(maxlen=HISTORY_LEN)
        self.ready      = False
        self.start_time = time.time()
        self._log_f  = open(log_path, 'w', newline='', buffering=1)
        self._writer = csv.DictWriter(self._log_f, fieldnames=self._CSV_COLS)
        self._writer.writeheader()
        print(f'Logging to {log_path}')

    # ── helpers ───────────────────────────────────────────────────────────────
    @property
    def best_bid(self): return max(self._bids) if self._bids else None
    @property
    def best_ask(self): return min(self._asks) if self._asks else None
    @property
    def mid(self):
        b, a = self.best_bid, self.best_ask
        return (b + a) / 2 if b and a else None

    @staticmethod
    def _exact(qty_str):
        return qty_str.rstrip('0').rstrip('.') == TARGET_SIZE_STR.rstrip('0').rstrip('.')

    def _ext(self, name):
        f = self.ext.get(name)
        return f if (f and f.fresh) else None

    def _ref(self):
        """Reference fair feed: Binance when it's alive, else Coinbase.
        (binance.com is geo-blocked from US IPs, so this is usually Coinbase.)"""
        return self._ext('binance') or self._ext('coinbase')

    def close(self):
        self._log_f.flush()
        self._log_f.close()

    def _log_event(self, e):
        mid  = e.get('mid')
        dist = e.get('dist')
        row = {c: '' for c in self._CSV_COLS}
        row['timestamp'] = e['t']
        row['elapsed_s'] = round(e['elapsed'], 4)
        row['side']      = e['side']
        row['event']     = e['event']
        row['price']     = e['price']
        for k in ('total_qty', 'fresh', 'at_bbo', 'dist_bbo_ticks'):
            if e.get(k) is not None:
                row[k] = e[k]
        if e.get('dur') is not None: row['duration_s'] = round(e['dur'], 4)
        if e.get('gap') is not None: row['gap_s'] = round(e['gap'], 4)
        if mid:
            row['mid'] = round(mid, 4)
            if dist is not None:
                row['dist_mid_abs'] = round(dist, 4)
                row['dist_mid_pct'] = round(dist / mid * 100, 6)
        if e.get('ext_venue') is not None: row['ext_venue'] = e['ext_venue']
        if e.get('rule_ok') is not None:   row['side_rule_ok'] = e['rule_ok']
        for k, col in (('bn_mid', 'bn_mid'), ('cb_mid', 'cb_mid'), ('basis', 'tm_ref_basis'),
                       ('peg_cb', 'peg_cb'),
                       ('edge_bn', 'edge_bn'), ('edge_cb', 'edge_cb'),
                       ('pre_ext', 'pre_drift_ext'), ('pre_toward', 'pre_toward_ext'),
                       ('life_ext', 'life_drift_ext'), ('toward', 'life_toward_ext')):
            if e.get(k) is not None:
                row[col] = round(e[k], 4)
        for h, col in zip(DRIFT_HORIZONS, ('drift_1s', 'drift_5s', 'drift_15s')):
            d = (e.get('drifts') or {}).get(h)
            if d is not None and not math.isnan(d):
                row[col] = round(d, 4)
        for h, col in zip(DRIFT_HORIZONS, ('ext_drift_1s', 'ext_drift_5s', 'ext_drift_15s')):
            d = (e.get('drifts_ext') or {}).get(h)
            if d is not None and not math.isnan(d):
                row[col] = round(d, 4)
        self._writer.writerow(row)

    # ── event firing ──────────────────────────────────────────────────────────
    def _fire_appear(self, side, price, now, fresh):
        """fresh=True: level was empty before (posted at its own level).
        fresh=False: level held something else and became 0.01 (model
        violation — residual, not a fresh post). fresh=None: from snapshot."""
        mid = self.mid
        if side == 'bid':
            best = self.best_bid
            ticks_behind = round((best - price) / TICK_SIZE) if best is not None else None
        else:
            best = self.best_ask
            ticks_behind = round((price - best) / TICK_SIZE) if best is not None else None
        at_bbo = ticks_behind == 0
        gap = (now - self.last_vanish[side]) if self.last_vanish[side] else None

        sgn = 1.0 if side == 'ask' else -1.0
        bn  = self._ext('binance')
        cb  = self._ext('coinbase')
        ref = bn or cb
        bn_mid  = bn.mid if bn else None
        cb_mid  = cb.mid if cb else None
        ref_mid = ref.mid if ref else None
        edge_bn = sgn * (price - bn_mid) if bn_mid else None
        edge_cb = sgn * (price - cb_mid) if cb_mid else None
        peg_cb  = (price - cb_mid) if cb_mid else None   # raw: 0 = posted AT Coinbase mid
        # side rule is judged against the TM mid EXCLUDING the order itself —
        # when it posts inside the spread it moves the BBO, and judging against
        # the moved mid would grade its own effect (it's alone at its level, so
        # dropping that one price level removes exactly it)
        if side == 'bid':
            others = [p for p in self._bids if abs(p - price) > 1e-9]
            ex_b, ex_a = (max(others) if others else None), self.best_ask
        else:
            others = [p for p in self._asks if abs(p - price) > 1e-9]
            ex_b, ex_a = self.best_bid, (min(others) if others else None)
        ex_mid = (ex_b + ex_a) / 2 if (ex_b is not None and ex_a is not None) else None
        rule_ok = None   # CB-peg side rule: ask iff CB above TM mid, bid iff below
        if cb_mid is not None and ex_mid is not None and abs(cb_mid - ex_mid) > 1e-9:
            rule_ok = (side == 'ask') == (cb_mid > ex_mid)
        pre_ext = ref.drift_since(now - PRE_WINDOW_SECS) if ref else None
        pre_toward = sgn * pre_ext if pre_ext is not None else None

        self.tracked[side][price] = {'t0': now, 'at_bbo': at_bbo, 'fresh': fresh,
                                     'mid0': mid, 'ref0': ref_mid,
                                     'ref_name': ref.name if ref else None,
                                     'alerted': False}
        if mid:
            self.pending_thesis.append({
                't0': now, 'side': side, 'price': price, 'mid0': mid,
                'ref0': ref_mid, 'ref_name': ref.name if ref else None,
                'at_bbo': at_bbo,
                'drifts':     {h: None for h in DRIFT_HORIZONS},
                'drifts_ext': {h: None for h in DRIFT_HORIZONS},
            })
        ev = {
            't': now, 'elapsed': now - self.start_time,
            'side': side, 'event': 'APPEAR', 'price': price,
            'total_qty': TARGET_SIZE, 'fresh': fresh, 'at_bbo': at_bbo,
            'dist_bbo_ticks': ticks_behind, 'gap': gap,
            'mid': mid, 'dist': (price - mid) if mid else None,
            'bn_mid': bn_mid, 'cb_mid': cb_mid,
            'ext_venue': ref.name if ref else None,
            'basis': (mid - ref_mid) if (mid and ref_mid) else None,
            'peg_cb': peg_cb, 'rule_ok': rule_ok,
            'edge_bn': edge_bn, 'edge_cb': edge_cb,
            'pre_ext': pre_ext, 'pre_toward': pre_toward,
        }
        self.events.appendleft(ev)
        self._log_event(ev)

    def _fire_gone(self, side, price, now, q, kind=None):
        info = self.tracked[side].pop(price, None)
        dur  = (now - info['t0']) if info else None
        mid  = self.mid
        if kind is None:
            # level wiped entirely: pulled, or taken whole. If the book crossed
            # through the price it was probably lifted/hit; otherwise a pull.
            if side == 'bid':
                crossed = self.best_ask is not None and price >= self.best_ask
            else:
                crossed = self.best_bid is not None and price <= self.best_bid
            kind = 'FILL?' if crossed else 'CANCEL?'
            self.last_vanish[side] = now

        sgn = 1.0 if side == 'ask' else -1.0
        f = self._ext(info['ref_name']) if (info and info.get('ref_name')) else None
        ref_mid = f.mid if f else None
        life_ext = toward = None
        if info and ref_mid is not None and info.get('ref0') is not None:
            life_ext = ref_mid - info['ref0']
            toward   = sgn * life_ext

        ev = {
            't': now, 'elapsed': now - self.start_time,
            'side': side, 'event': kind, 'price': price,
            'total_qty': q, 'dur': dur,
            'at_bbo': info['at_bbo'] if info else None,
            'mid': mid, 'dist': (price - mid) if mid else None,
            'ext_venue': info.get('ref_name') if info else None,
            'life_ext': life_ext, 'toward': toward,
        }
        self.events.appendleft(ev)
        self._log_event(ev)

    # ── message handlers ──────────────────────────────────────────────────────
    def handle_snapshot(self, data):
        now = time.time()
        self._bids = {}
        self._asks = {}
        self.tracked['bid'].clear()
        self.tracked['ask'].clear()
        # build the full book first — at_bbo needs the complete snapshot
        for b in data.get('bids', []):
            p, q = float(b['price']), float(b['qty'])
            if q > 0:
                self._bids[p] = q
        for a in data.get('asks', []):
            p, q = float(a['price']), float(a['qty'])
            if q > 0:
                self._asks[p] = q
        for b in data.get('bids', []):
            if float(b['qty']) > 0 and self._exact(b['qty']):
                self._fire_appear('bid', float(b['price']), now, fresh=None)
        for a in data.get('asks', []):
            if float(a['qty']) > 0 and self._exact(a['qty']):
                self._fire_appear('ask', float(a['price']), now, fresh=None)
        self.ready = True
        self._record_series(now)

    def handle_update(self, data):
        now = time.time()
        for b in data.get('bids', []):
            p, q_str = float(b['price']), b['qty']
            q = float(q_str)
            old = self._bids.get(p, 0.0)
            if q == 0: self._bids.pop(p, None)
            else:      self._bids[p] = q
            self._process_level('bid', p, q_str, q, old, now)
        for a in data.get('asks', []):
            p, q_str = float(a['price']), a['qty']
            q = float(q_str)
            old = self._asks.get(p, 0.0)
            if q == 0: self._asks.pop(p, None)
            else:      self._asks[p] = q
            self._process_level('ask', p, q_str, q, old, now)
        self._record_series(now)

    def _process_level(self, side, price, q_str, q, old, now):
        tracked_here = price in self.tracked[side]
        if q > 0 and self._exact(q_str):
            # level shows exactly 0.01 — per the model, that IS the target
            if not tracked_here:
                self._fire_appear(side, price, now, fresh=(old <= SIZE_TOL))
        elif tracked_here:
            if q <= 0:
                self._fire_gone(side, price, now, 0.0)               # pull or full take
            elif q > TARGET_SIZE + SIZE_TOL:
                self._fire_gone(side, price, now, q, kind='JOINED')  # model violation
            else:
                self._fire_gone(side, price, now, q, kind='PARTIAL') # model violation

    # ── series + thesis drift ─────────────────────────────────────────────────
    def _record_series(self, now):
        mid = self.mid
        bn  = self._ext('binance')
        cb  = self._ext('coinbase')
        ref = bn or cb
        ref_mid = ref.mid if ref else None
        tb = max(self.tracked['bid']) if self.tracked['bid'] else float('nan')
        ta = min(self.tracked['ask']) if self.tracked['ask'] else float('nan')
        self.ts.append(now - self.start_time)
        self.mids.append(mid or float('nan'))
        self.bn_mids.append((bn.mid if bn else None) or float('nan'))
        self.cb_mids.append((cb.mid if cb else None) or float('nan'))
        self.basis.append((mid - ref_mid) if (mid and ref_mid) else float('nan'))
        self.t_bid_px.append(tb)
        self.t_ask_px.append(ta)
        self._check_thesis(now)
        self._check_stale(now)

    def _check_thesis(self, now):
        mid = self.mid
        matured = []
        for rec in self.pending_thesis:
            f = self._ext(rec['ref_name']) if rec.get('ref_name') else None
            ref_mid_now = f.mid if f else None
            for h in DRIFT_HORIZONS:
                if rec['drifts'][h] is None and now - rec['t0'] >= h:
                    rec['drifts'][h] = (mid - rec['mid0']) if mid else float('nan')
                    rec['drifts_ext'][h] = ((ref_mid_now - rec['ref0'])
                                            if (ref_mid_now is not None and rec['ref0'] is not None)
                                            else float('nan'))
            if all(v is not None for v in rec['drifts'].values()):
                matured.append(rec)
        for rec in matured:
            self.pending_thesis.remove(rec)
            self.thesis_done.append(rec)
            ev = {
                't': now, 'elapsed': now - self.start_time,
                'side': rec['side'], 'event': 'THESIS', 'price': rec['price'],
                'at_bbo': rec['at_bbo'], 'mid': rec['mid0'],
                'ext_venue': rec.get('ref_name'),
                'drifts': rec['drifts'], 'drifts_ext': rec['drifts_ext'],
            }
            self.events.appendleft(ev)
            self._log_event(ev)

    def _check_stale(self, now):
        """Abandoned-order alarm: they normally cancel within ~5s, so a tracked
        0.01 that is both old AND mispriced vs Coinbase is a broken-cancel
        leftover — directly exploitable (see main.ipynb's stale snipe)."""
        cb = self._ext('coinbase')
        cb_mid = cb.mid if cb else None
        if cb_mid is None:
            return
        for side, levels in self.tracked.items():
            for price, info in levels.items():
                if info.get('alerted'):
                    continue
                age  = now - info['t0']
                edge = (price - cb_mid) if side == 'bid' else (cb_mid - price)
                if age >= STALE_AFTER_SECS and edge >= STALE_EDGE_USD:
                    info['alerted'] = True
                    ev = {
                        't': now, 'elapsed': now - self.start_time,
                        'side': side, 'event': 'STALE', 'price': price,
                        'dur': age, 'at_bbo': info.get('at_bbo'),
                        'mid': self.mid, 'dist': (price - self.mid) if self.mid else None,
                        'cb_mid': cb_mid, 'peg_cb': price - cb_mid,
                    }
                    self.events.appendleft(ev)
                    self._log_event(ev)
                    print(f'⚠ STALE 0.01 {side} @ {price:.1f} — {age:.0f}s old, '
                          f'{edge:+.2f} vs CB mid {cb_mid:.1f}: abandoned order, exploitable')

    @staticmethod
    def _signed(rec, h, key='drifts'):
        """Drift signed so positive = market moved AGAINST the exposed order."""
        d = rec[key][h]
        if d is None or math.isnan(d):
            return None
        return d if rec['side'] == 'ask' else -d

    # ── aggregates ────────────────────────────────────────────────────────────
    def stats(self):
        appears  = [e for e in self.events if e['event'] == 'APPEAR']
        fills    = [e for e in self.events if e['event'] == 'FILL?']
        cancels  = [e for e in self.events if e['event'] == 'CANCEL?']
        joins    = [e for e in self.events if e['event'] == 'JOINED']
        partials = [e for e in self.events if e['event'] == 'PARTIAL']
        durs = [e['dur'] for e in self.events
                if e['event'] in ('FILL?', 'CANCEL?') and e.get('dur')]
        gaps = [e['gap'] for e in appears if e.get('gap')]

        def _drift_stats(key):
            out = {}
            for h in DRIFT_HORIZONS:
                vals = [s for rec in self.thesis_done
                        if (s := self._signed(rec, h, key)) is not None]
                out[h] = {'n': len(vals),
                          'hit': (sum(1 for v in vals if v > 0) / len(vals)) if vals else None,
                          'avg': (sum(vals) / len(vals)) if vals else None}
            return out

        def _agg(vals):
            return {'n': len(vals),
                    'avg': (sum(vals) / len(vals)) if vals else None,
                    'med': statistics.median(vals) if vals else None,
                    'frac_pos': (sum(1 for v in vals if v > 0) / len(vals)) if vals else None}

        pegs  = [e['peg_cb'] for e in appears if e.get('peg_cb') is not None]
        rules = [e['rule_ok'] for e in appears if e.get('rule_ok') is not None]
        close = PEG_CLOSE_TICKS * TICK_SIZE

        n_app = len(appears)
        return {
            'appears':     n_app,
            'bid_appears': sum(1 for e in appears if e['side'] == 'bid'),
            'ask_appears': sum(1 for e in appears if e['side'] == 'ask'),
            'at_bbo_frac': (sum(1 for e in appears if e.get('at_bbo')) / n_app) if n_app else 0.0,
            'fresh_frac':  (sum(1 for e in appears if e.get('fresh')) / n_app) if n_app else 0.0,
            'fills': len(fills), 'cancels': len(cancels),
            'joined': len(joins), 'partials': len(partials),
            'avg_dur': (sum(durs) / len(durs)) if durs else 0.0,
            'med_dur': statistics.median(durs) if durs else 0.0,
            'med_gap': statistics.median(gaps) if gaps else 0.0,
            'thesis':     _drift_stats('drifts'),
            'thesis_ext': _drift_stats('drifts_ext'),
            'peg': {'n': len(pegs),
                    'med_abs': statistics.median([abs(p) for p in pegs]) if pegs else None,
                    'frac_close': (sum(1 for p in pegs if abs(p) <= close) / len(pegs)) if pegs else None},
            'rule': {'n': len(rules),
                     'hit': (sum(1 for r in rules if r) / len(rules)) if rules else None},
            'edge_bid': _agg([e['edge_cb'] for e in appears
                              if e['side'] == 'bid' and e.get('edge_cb') is not None]),
            'edge_ask': _agg([e['edge_cb'] for e in appears
                              if e['side'] == 'ask' and e.get('edge_cb') is not None]),
            'post_toward': _agg([e['pre_toward'] for e in appears
                                 if e.get('pre_toward') is not None]),
            'pull_toward': _agg([e['toward'] for e in cancels
                                 if e.get('toward') is not None]),
            'currently_bid': sorted(self.tracked['bid'], reverse=True),
            'currently_ask': sorted(self.tracked['ask']),
        }

    def level_table(self):
        """
        Per-level breakdown near the spread.
        Each row includes:
          dist_mid_abs / pct  — distance from mid
          dist_next_abs / pct — gap to the next better order on the same side
                                (0 if it IS the best level)
        """
        mid  = self.mid or 0
        all_bids = sorted(self._bids.keys(), reverse=True)  # high → low
        all_asks = sorted(self._asks.keys())                 # low  → high

        bids = [(p, self._bids[p]) for p in all_bids[:LEVELS_DISPLAY]]
        asks = [(p, self._asks[p]) for p in all_asks[:LEVELS_DISPLAY]]

        def _row(side, p, q, better_px):
            has_t = p in self.tracked[side]
            other = round(q - TARGET_SIZE, 8) if has_t else None
            dist_mid_abs = p - mid
            dist_mid_pct = (dist_mid_abs / mid * 100) if mid else 0.0
            if better_px is not None:
                dist_next_abs = abs(p - better_px)
                dist_next_pct = (dist_next_abs / better_px * 100) if better_px else 0.0
            else:
                dist_next_abs = 0.0
                dist_next_pct = 0.0
            return (side, p, q, has_t, other,
                    dist_mid_abs, dist_mid_pct,
                    dist_next_abs, dist_next_pct)

        rows = []
        for i, (p, q) in enumerate(reversed(asks)):
            idx_in_all = all_asks.index(p)
            better_px  = all_asks[idx_in_all - 1] if idx_in_all > 0 else None
            rows.append(_row('ask', p, q, better_px))
        for i, (p, q) in enumerate(bids):
            idx_in_all = all_bids.index(p)
            better_px  = all_bids[idx_in_all - 1] if idx_in_all > 0 else None
            rows.append(_row('bid', p, q, better_px))
        return rows

In [ ]:
def render(tracker):
    clear_output(wait=True)

    s   = tracker.stats()
    ts  = list(tracker.ts)

    fig = plt.figure(figsize=(14, 12))
    gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.35)
    ax_price = fig.add_subplot(gs[0, :])
    ax_dur   = fig.add_subplot(gs[1, 0])
    ax_side  = fig.add_subplot(gs[1, 1])
    ax_th    = fig.add_subplot(gs[1, 2])
    ax_peg   = fig.add_subplot(gs[2, 0])
    ax_flow  = fig.add_subplot(gs[2, 1])
    ax_basis = fig.add_subplot(gs[2, 2])

    # ── prices: TM mid vs external fairs, with the 0.01's postings ───────────
    ax_price.plot(ts, list(tracker.mids),    color='gray',       linewidth=1, label='TM mid', alpha=0.8)
    ax_price.plot(ts, list(tracker.bn_mids), color='goldenrod',  linewidth=1, label='Binance mid', alpha=0.7)
    ax_price.plot(ts, list(tracker.cb_mids), color='deepskyblue',linewidth=1, label='Coinbase mid', alpha=0.7)
    ax_price.scatter(ts, list(tracker.t_bid_px), color='lime',   s=8, label='0.01 bid', zorder=3)
    ax_price.scatter(ts, list(tracker.t_ask_px), color='salmon', s=8, label='0.01 ask', zorder=3)
    cur_b = f"{s['currently_bid'][0]:.1f}" if s['currently_bid'] else '—'
    cur_a = f"{s['currently_ask'][0]:.1f}" if s['currently_ask'] else '—'
    ax_price.set_title(f'Target ({TARGET_SIZE_STR} BTC)  |  bid={cur_b}  ask={cur_a}')
    ax_price.set_ylabel('Price ($)'); ax_price.legend(loc='upper left', fontsize=8, ncol=2)
    ax_price.grid(True, alpha=0.3)

    # ── exposure durations ────────────────────────────────────────────────────
    durs = [e['dur'] for e in tracker.events
            if e['event'] in ('FILL?', 'CANCEL?') and e.get('dur', 0) > 0]
    if durs:
        ax_dur.hist(durs, bins=30, color='steelblue', alpha=0.75, edgecolor='white')
        ax_dur.axvline(s['avg_dur'], color='orange', linestyle='--', label=f"avg {s['avg_dur']:.1f}s")
        ax_dur.axvline(s['med_dur'], color='red', linestyle=':',  label=f"med {s['med_dur']:.1f}s")
        ax_dur.legend(fontsize=8)
    ax_dur.set_title('Exposure Duration (s)'); ax_dur.set_xlabel('seconds')
    ax_dur.grid(True, alpha=0.3)

    # ── event counts ──────────────────────────────────────────────────────────
    ax_side.bar(['Bid', 'Ask', 'Fill?', 'Cancel?', 'Joined', 'Partial'],
                [s['bid_appears'], s['ask_appears'], s['fills'],
                 s['cancels'], s['joined'], s['partials']],
                color=['lime', 'salmon', 'gold', 'steelblue', 'gray', 'orange'])
    ax_side.set_title('Event Counts'); ax_side.grid(True, alpha=0.3, axis='y')
    ax_side.tick_params(axis='x', labelsize=8)

    # ── thesis: drift AGAINST the order after APPEAR (TM vs external terms) ──
    labels = [f'{h:g}s' for h in DRIFT_HORIZONS]
    x = range(len(labels))
    tm_avgs  = [s['thesis'][h]['avg']     or 0.0 for h in DRIFT_HORIZONS]
    ext_avgs = [s['thesis_ext'][h]['avg'] or 0.0 for h in DRIFT_HORIZONS]
    ax_th.bar([i - 0.2 for i in x], tm_avgs,  width=0.4, color='gray',        label='TM mid')
    ax_th.bar([i + 0.2 for i in x], ext_avgs, width=0.4, color='deepskyblue', label='Ext fair')
    ax_th.axhline(0, color='black', linewidth=1)
    ax_th.set_xticks(list(x)); ax_th.set_xticklabels(labels)
    for i, h in enumerate(DRIFT_HORIZONS):
        t = s['thesis'][h]
        if t['hit'] is not None:
            ax_th.annotate(f"{t['hit']:.0%}", (i - 0.2, tm_avgs[i]),
                           ha='center', va='bottom', fontsize=8)
    ax_th.set_title('Avg drift AGAINST the 0.01 ($)')
    ax_th.legend(fontsize=8); ax_th.grid(True, alpha=0.3, axis='y')

    # ── CB-peg hypothesis: posting price − Coinbase mid ───────────────────────
    appears = [e for e in tracker.events if e['event'] == 'APPEAR']
    pb = [e['peg_cb'] for e in appears if e['side'] == 'bid' and e.get('peg_cb') is not None]
    pa = [e['peg_cb'] for e in appears if e['side'] == 'ask' and e.get('peg_cb') is not None]
    if pb or pa:
        if pb: ax_peg.hist(pb, bins=25, color='lime',   alpha=0.6, label=f'bid (n={len(pb)})')
        if pa: ax_peg.hist(pa, bins=25, color='salmon', alpha=0.6, label=f'ask (n={len(pa)})')
        ax_peg.axvline(0, color='black', linewidth=1)
        ax_peg.legend(fontsize=8)
    peg, rule = s['peg'], s['rule']
    peg_note  = (f"med |peg| {peg['med_abs']:.2f}  within {PEG_CLOSE_TICKS} ticks {peg['frac_close']:.0%}"
                 if peg['med_abs'] is not None else 'no CB data')
    rule_note = f"side rule {rule['hit']:.0%} (n={rule['n']})" if rule['hit'] is not None else 'side rule —'
    ax_peg.set_title(f'APPEAR price − Coinbase mid ($)\n{peg_note}  |  {rule_note}')
    ax_peg.set_xlabel('$'); ax_peg.grid(True, alpha=0.3)

    # ── does fair drive it? post after fair leaves, pull as fair approaches ──
    po, pu = s['post_toward'], s['pull_toward']
    vals  = [po['avg'] or 0.0, pu['avg'] or 0.0]
    cols  = ['green' if v > 0 else 'red' for v in vals]
    bars  = ax_flow.bar(['pre-APPEAR\n(1s before)', 'over life\n(at CANCEL?)'], vals, color=cols, alpha=0.8)
    for bar, agg in zip(bars, (po, pu)):
        note = f"n={agg['n']}" + (f"\n{agg['frac_pos']:.0%} toward" if agg['frac_pos'] is not None else '')
        ax_flow.annotate(note, (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                         ha='center', va='bottom', fontsize=8)
    ax_flow.axhline(0, color='black', linewidth=1)
    ax_flow.set_title('Ext-fair drift TOWARD the order ($)')
    ax_flow.grid(True, alpha=0.3, axis='y')
    ax_flow.tick_params(axis='x', labelsize=8)

    # ── TM vs reference-fair basis ────────────────────────────────────────────
    ax_basis.plot(ts, list(tracker.basis), color='purple', linewidth=1)
    ax_basis.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax_basis.set_title('TM mid − ext fair ($)')
    ax_basis.grid(True, alpha=0.3)

    hit5     = s['thesis'][5.0]['hit']
    hit5_ext = s['thesis_ext'][5.0]['hit']
    peg_str  = f"|{peg['med_abs']:.2f}| {peg['frac_close']:.0%}" if peg['med_abs'] is not None else '—'
    rule_str = f"{rule['hit']:.0%}" if rule['hit'] is not None else '—'
    tm_str   = f'{hit5:.0%}' if hit5 is not None else '—'
    ext_str  = f'{hit5_ext:.0%}' if hit5_ext is not None else '—'
    plt.suptitle(
        f"appears={s['appears']}  at_bbo={s['at_bbo_frac']:.0%}  own_level={s['fresh_frac']:.0%}  "
        f"med_dur={s['med_dur']:.1f}s  med_repost={s['med_gap']:.1f}s  "
        f"CB_peg={peg_str}  side_rule={rule_str}  "
        f"thesis@5s TM={tm_str} EXT={ext_str}  "
        f"violations={s['joined'] + s['partials']}",
        fontsize=10
    )
    display(fig); plt.close(fig)

    # ── per-level breakdown ───────────────────────────────────────────────────
    H  = f"  {'':4}  {'price':>9}  {'total qty':>10}  {'target':>8}  {'other':>10}"
    H += f"  {'mid abs':>8}  {'mid%':>6}  {'next abs':>9}  {'next%':>6}"
    print(H)
    print('  ' + '-' * (len(H) - 2))
    for row in tracker.level_table():
        side, p, q, has_t, other, d_mid_a, d_mid_p, d_nxt_a, d_nxt_p = row
        arrow  = 'ASK' if side == 'ask' else 'BID'
        flag   = f'✓{TARGET_SIZE_STR}' if has_t else ''
        other_s = f'{other:.8f}'.rstrip('0') if other is not None and other > 0 else ''
        d_mid_str = f'{d_mid_a:+.2f}' if d_mid_a else '  —'
        d_mid_pct = f'{d_mid_p:+.4f}%' if d_mid_p else '  —'
        d_nxt_str = f'{d_nxt_a:.2f}' if d_nxt_a else '  —'
        d_nxt_pct = f'{d_nxt_p:.4f}%' if d_nxt_p else '  —'
        print(f'  {arrow:>4}  {p:>9.1f}  {q:>10.8f}  {flag:>8}  {other_s:>10}'
              f'  {d_mid_str:>8}  {d_mid_pct:>6}  {d_nxt_str:>9}  {d_nxt_pct:>6}')

    # ── event log ─────────────────────────────────────────────────────────────
    print(f"\n{'time':>8}  {'side':>4}  {'event':>8}  {'price':>9}  {'bbo':>4}  {'dur':>7}  note")
    print('-' * 100)
    for e in list(tracker.events)[:EVENT_DISPLAY]:
        t_str = f"{e['elapsed']:7.1f}s"
        bbo   = 'yes' if e.get('at_bbo') else ('no' if e.get('at_bbo') is not None else '  —')
        dur   = f"{e['dur']:.2f}s" if e.get('dur') else '      —'
        if e['event'] == 'THESIS':
            parts = []
            for h in DRIFT_HORIZONS:
                d = e['drifts'].get(h)
                if d is not None and not math.isnan(d):
                    signed = d if e['side'] == 'ask' else -d
                    parts.append(f'{h:g}s:{signed:+.2f}')
            note = 'against-drift  ' + '  '.join(parts)
        elif e['event'] == 'APPEAR':
            bits = []
            if e.get('dist_bbo_ticks') is not None:
                bits.append('at BBO' if e['dist_bbo_ticks'] == 0
                            else f"{e['dist_bbo_ticks']} tick(s) behind best")
            if e.get('peg_cb') is not None:
                bits.append(f"CB peg {e['peg_cb']:+.2f}")
            if e.get('rule_ok') is not None:
                bits.append('✓rule' if e['rule_ok'] else '✗rule')
            if e.get('pre_toward') is not None:
                bits.append(f"fair pre-drift {e['pre_toward']:+.2f} toward")
            if e.get('gap'):
                bits.append(f"repost after {e['gap']:.1f}s")
            if e.get('fresh') is not None:
                bits.append('own level' if e['fresh'] else 'existing level!')
            note = ', '.join(bits)
        elif e['event'] == 'STALE':
            edge = e['peg_cb'] if e['side'] == 'bid' else -e['peg_cb']
            note = f"⚠ abandoned: {e['dur']:.0f}s old, {edge:+.2f} vs CB — exploitable"
        elif e['event'] in ('JOINED', 'PARTIAL'):
            note = f"level now {e.get('total_qty', 0):.8f} — model violation"
        elif e.get('toward') is not None:
            note = f"fair moved {e['toward']:+.2f} toward it over its life"
        else:
            note = ''
        print(f"{t_str}  {e['side']:>4}  {e['event']:>8}  {e['price']:>9.1f}  {bbo:>4}  {dur:>7}  {note}")

In [ ]:
async def stream():
    log_path = LOG_DIR / f"signal_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    binance  = BinanceFeed()
    coinbase = CoinbaseFeed()
    tracker  = SignalTracker(log_path, ext={'binance': binance, 'coinbase': coinbase})
    feed_tasks = [asyncio.create_task(binance.run()),
                  asyncio.create_task(coinbase.run())]
    headers  = {
        'Origin': 'https://truemarkets.co',
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36',
        'Accept-Language': 'en-US,en;q=0.9',
    }
    RENDER_EVERY = 5.0
    last_render  = 0.0

    print(f'Connecting to {WS_URL}...')
    backoff = 1
    try:
        while True:
            try:
                async with websockets.connect(WS_URL, additional_headers=headers) as ws:
                    backoff = 1
                    await ws.send(json.dumps({
                        'type': 'SUBSCRIBE_NO_AUTH',
                        'item_names': [SYMBOL],
                        'channels': ['DEPTH'],
                        'timestamp': str(int(time.time())),
                    }))
                    print(f'Subscribed. Watching for the {TARGET_SIZE_STR} BTC order...')

                    async for raw in ws:
                        try: msg = json.loads(raw)
                        except Exception: continue

                        t = msg.get('update')
                        d = msg.get('data', {})
                        if   t == 'SNAPSHOT': tracker.handle_snapshot(d)
                        elif t == 'UPDATE':   tracker.handle_update(d)
                        else: continue

                        now = time.time()
                        if now - last_render >= RENDER_EVERY:
                            render(tracker)
                            last_render = now

            except websockets.ConnectionClosed:
                print('Connection closed, reconnecting...')
            except asyncio.CancelledError:
                raise
            except Exception as e:
                print(f'Error: {e}')
            await asyncio.sleep(backoff)
            backoff = min(backoff * 2, 30)

    except asyncio.CancelledError:
        pass
    finally:
        for t in feed_tasks:
            t.cancel()
        tracker.close()
        print(f'\nStopped. Log saved to {log_path}')

# Interrupt kernel to stop
await stream()